In [6]:
from google.colab import files
uploaded = files.upload()


Saving symptom.csv to symptom.csv


In [7]:
import pandas as pd

df = pd.read_csv("symptom.csv")

print(df.head())

                                               text               label
0      symptoms include olive green spots on leaves  Apple___Apple_scab
1     visible signs are olive green spots on leaves  Apple___Apple_scab
2             plant has olive green spots on leaves  Apple___Apple_scab
3  the plant is showing olive green spots on leaves  Apple___Apple_scab
4                 i see olive green spots on leaves  Apple___Apple_scab


In [8]:
print(df.shape)
print(df.columns)

(1710, 2)
Index(['text', 'label'], dtype='object')


In [9]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [10]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

In [11]:
import torch
from torch.utils.data import Dataset

In [12]:
from sklearn.metrics import accuracy_score

In [13]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

In [14]:
df = pd.read_csv("symptom.csv")

print(df.head())

                                               text               label
0      symptoms include olive green spots on leaves  Apple___Apple_scab
1     visible signs are olive green spots on leaves  Apple___Apple_scab
2             plant has olive green spots on leaves  Apple___Apple_scab
3  the plant is showing olive green spots on leaves  Apple___Apple_scab
4                 i see olive green spots on leaves  Apple___Apple_scab


In [15]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])

In [16]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [17]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [18]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=128
)

In [19]:
class PlantDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = PlantDataset(train_encodings, train_labels)
test_dataset = PlantDataset(test_encodings, test_labels)

In [20]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(le.classes_)
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
!pip install -U transformers

In [43]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    save_strategy="epoch"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [44]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [45]:
trainer.train()

Step,Training Loss
500,0.802581


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=855, training_loss=0.6963335896096035, metrics={'train_runtime': 135.3618, 'train_samples_per_second': 50.531, 'train_steps_per_second': 6.316, 'total_flos': 45709760285280.0, 'train_loss': 0.6963335896096035, 'epoch': 5.0})

In [46]:
from sklearn.metrics import accuracy_score

predictions = trainer.predict(test_dataset)

preds = predictions.predictions.argmax(axis=1)
labels = predictions.label_ids

accuracy = accuracy_score(labels, preds)
print("Accuracy:", accuracy)

Accuracy: 0.7485380116959064


In [47]:
df = pd.read_csv("symptom.csv")

# SAVE ORIGINAL LABELS
original_labels = df["label"].copy()

le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])

In [48]:
def predict(text):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    outputs = model(**inputs)
    logits = outputs.logits

    predicted_class_id = logits.argmax().item()

    return le.inverse_transform([predicted_class_id])[0]

In [49]:
print(predict("yellow spots on leaves"))
print(predict("white powder on leaves"))

Tomato___Leaf_Mold
Squash___Powdery_mildew


In [51]:
def chatbot():
    while True:
        text = input("Enter symptoms (or 'exit'): ")
        if text == "exit":
            break
        print("Predicted Disease:", predict(text))

chatbot()